# Meteostat

In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
import folium
from meteostat import Stations, Daily, Point
from datetime import datetime
import time
import re
from unidecode import unidecode
import matplotlib.pyplot as plt

In [2]:
# Initialize geocoder
geolocator = Nominatim(user_agent="weather_locator")

## Clean Cities

In [3]:
df_dengue = pd.read_csv('silver/dengue_city_geolocated.csv')
df_dengue.head()

,SEMANA,ANO,country,state,city,count,idx_city,Latitude,Longitude,date,week
0,1,2007,colombia,arauca,arauca,2,COLOMBIA_ARAUCA_ARAUCA,7.085199,-70.757776,2007-01-07,07-01-2007
1,1,2007,colombia,arauca,saravena,1,COLOMBIA_ARAUCA_SARAVENA,6.906819,-71.836133,2007-01-07,07-01-2007
2,1,2007,colombia,atlantico,baranoa,7,COLOMBIA_ATLANTICO_BARANOA,10.793582,-74.915216,2007-01-07,07-01-2007
3,1,2007,colombia,atlantico,barranquilla,23,COLOMBIA_ATLANTICO_BARRANQUILLA,11.010192,-74.823179,2007-01-07,07-01-2007
4,1,2007,colombia,atlantico,galapa,2,COLOMBIA_ATLANTICO_GALAPA,10.896866,-74.885961,2007-01-07,07-01-2007


In [4]:
df_dengue_og = df_dengue.copy()

In [5]:
df_dengue = df_dengue[df_dengue['ANO'] == 2023]
df_dengue.drop(columns = ['SEMANA', 'ANO', 'week'], inplace=True)
df_dengue['date'] = pd.to_datetime(df_dengue['date'])

In [6]:
df_dengue

,country,state,city,count,idx_city,Latitude,Longitude,date
131554,aruba,exterior,exterior_aruba,1,ARUBA_EXTERIOR_EXTERIOR_ARUBA,12.501363,-69.961848,2023-01-08
131555,brasil,exterior,exterior_brasil,1,BRASIL_EXTERIOR_EXTERIOR_BRASIL,-10.333333,-53.200000,2023-01-08
131556,colombia,amazonas,leticia,7,COLOMBIA_AMAZONAS_LETICIA,-4.212921,-69.942596,2023-01-08
131557,colombia,antioquia,apartado,7,COLOMBIA_ANTIOQUIA_APARTADO,7.884901,-76.622746,2023-01-08
131558,colombia,antioquia,caracoli,1,COLOMBIA_ANTIOQUIA_CARACOLI,6.409276,-74.756698,2023-01-08
...,...,...,...,...,...,...,...,...
147043,colombia,vaupes,mitu,8,COLOMBIA_VAUPES_MITU,1.253850,-70.234558,2023-12-31
147044,colombia,vichada,puerto carreno,1,COLOMBIA_VICHADA_PUERTO CARREÑO,6.190923,-67.484189,2023-12-31
147045,comoras,exterior,exterior_comoras,1,COMORAS_EXTERIOR_EXTERIOR_COMORAS,-12.204518,44.283296,2023-12-31
147046,haiti,exterior,exterior_haiti,1,HAITÍ_EXTERIOR_EXTERIOR_HAITÍ,19.139995,-72.357097,2023-12-31


In [7]:
df = df_dengue.copy()

In [8]:
# Function to clean text
def clean_text(text):
    text = str(text).lower()               # Lowercase
    text = unidecode(text)                 # Remove accents
    text = re.sub(r'[^\w\s]', '', text)    # Remove punctuation
    text = re.sub(r'\s+', ' ', text)       # Normalize whitespace
    return text.strip()

# Apply to relevant columns
# Apply to relevant columns
df['country'] = df['country'].apply(clean_text)
df['state'] = df['state'].apply(clean_text)
df['city'] = df['city'].apply(clean_text)

# Optional: remove duplicate cities (if any)
df.drop_duplicates(subset=['city', 'state', 'country'], inplace=True)

In [9]:
df

,country,state,city,count,idx_city,Latitude,Longitude,date
131554,aruba,exterior,exterior_aruba,1,ARUBA_EXTERIOR_EXTERIOR_ARUBA,12.501363,-69.961848,2023-01-08
131555,brasil,exterior,exterior_brasil,1,BRASIL_EXTERIOR_EXTERIOR_BRASIL,-10.333333,-53.200000,2023-01-08
131556,colombia,amazonas,leticia,7,COLOMBIA_AMAZONAS_LETICIA,-4.212921,-69.942596,2023-01-08
131557,colombia,antioquia,apartado,7,COLOMBIA_ANTIOQUIA_APARTADO,7.884901,-76.622746,2023-01-08
131558,colombia,antioquia,caracoli,1,COLOMBIA_ANTIOQUIA_CARACOLI,6.409276,-74.756698,2023-01-08
...,...,...,...,...,...,...,...,...
146890,colombia,huila,teruel,1,COLOMBIA_HUILA_TERUEL,2.741633,-75.568345,2023-12-31
146951,colombia,quindio,buenavista,1,COLOMBIA_QUINDIO_BUENAVISTA,4.359948,-75.738723,2023-12-31
146956,colombia,risaralda,belen de umbria,1,COLOMBIA_RISARALDA_BELEN DE UMBRIA,5.200909,-75.868993,2023-12-31
147045,comoras,exterior,exterior_comoras,1,COMORAS_EXTERIOR_EXTERIOR_COMORAS,-12.204518,44.283296,2023-12-31


# Geocode

In [10]:
# Function to geocode city
def geocode_city(city, depto, country):
    query = f"{city}, {depto}, {country}"
    try:
        location = geolocator.geocode(query)
        if location:
            return location.latitude, location.longitude
    except Exception as e:
        print(f"Geocoding error for {query}: {e}")
    return None, None

# Meteo

In [11]:
cities_df = df[['city', 'state', 'country', 'Latitude', 'Longitude', 'idx_city']].copy().drop_duplicates().reset_index(drop=True)

In [12]:
cities_df.dropna(subset = ['Latitude', 'Longitude'], inplace=True)

In [13]:
cities_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 802 entries, 0 to 810
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   city       802 non-null    object 
 1   state      802 non-null    object 
 2   country    802 non-null    object 
 3   Latitude   802 non-null    float64
 4   Longitude  802 non-null    float64
 5   idx_city   802 non-null    object 
dtypes: float64(2), object(4)
memory usage: 43.9+ KB


In [29]:
cities_df.to_csv('meteo_colombia/cities_geolocated.csv', index=False)

In [14]:
# Define the date range
start = datetime(2023, 1, 1)
end = datetime(2023, 12, 31)

In [15]:
# create dir if not exist 'meteo_colombia/cities/'
import os
if not os.path.exists('meteo_colombia/cities/'):
    os.makedirs('meteo_colombia/cities/')


In [16]:
import time
from tqdm import tqdm

In [23]:
cities_df.reset_index(drop=True, inplace=True)

In [24]:
cities_df[84:85]

,city,state,country,Latitude,Longitude,idx_city
84,lloro,choco,colombia,5.584484,-76.372565,COLOMBIA_CHOCO_LLORO


In [25]:
range(84,len(cities_df))

range(84, 802)

In [27]:
# got error at 84, go from 84 to the end
for i in tqdm(range(84,len(cities_df))):
    
    lat = cities_df['Latitude'][i]
    lon = cities_df['Longitude'][i]
    # if lat or lon is None, skip the city
    if lat is None or lon is None:
        print(f"Skipping {cities_df['City'][i]} due to missing coordinates.")
        continue
    # create point for city
    city = Point(lat, lon)

    # Get daily data for
    data = Daily(city, start, end)
    data = data.fetch()
    # save data to csv in 'meteo_colombia/cities/' folder
    data.to_csv(f'meteo_colombia/cities/{cities_df["idx_city"][i]}.csv')
    time.sleep(5)
    

100%|██████████| 718/718 [1:01:13<00:00,  5.12s/it]
